# Question 1: Reproducible ML Experiment with Git, DVC and CML

This experiment trains an Iris classifier, reports test accuracy and weighted F1, and documents the script, DVC, Git, and CML workflow. The reusable pipeline implementation is in `src/`, with configuration in `params.yaml` and `dvc.yaml`.

Install project dependencies before running: `pip install -r requirements.txt`.

## 1. Setup and Imports

Load the libraries used for dataset inspection, training, scoring, and artifact output. The pinned project dependencies are declared in `requirements.txt`; avoid installing packages during a normal notebook run.

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import sklearn
import yaml
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
print(f"scikit-learn {sklearn.__version__}; NumPy {np.__version__}; pandas {pd.__version__}")

## 2. Load and Explore the Iris Dataset

Iris is bundled with scikit-learn, so no network download is needed. Inspect dimensions, feature names, target labels, and per-class counts before splitting.

In [ ]:
iris = load_iris(as_frame=True)
dataset = iris.frame.copy()
dataset["target_name"] = dataset["target"].map(dict(enumerate(iris.target_names)))
print("Dataset shape:", dataset.shape)
print("Features:", iris.feature_names)
print("Target labels:", list(iris.target_names))
print("Class distribution:\n", dataset["target_name"].value_counts().sort_index())
dataset.head()

## 3. Train a Baseline Model

Use a stratified 80/20 split and a standardized Logistic Regression model. The fixed random seed makes the split and estimator repeatable.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.20, random_state=RANDOM_STATE, stratify=iris.target
)
model = LogisticRegression(max_iter=500, random_state=RANDOM_STATE)
model.fit(x_train, y_train)
print(f"Trained on {len(x_train)} rows; held out {len(x_test)} rows.")

## 4. Evaluate Model Metrics

Report held-out accuracy and weighted F1, along with per-class precision/recall and a confusion matrix.

In [ ]:
y_pred = model.predict(x_test)
metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "f1_weighted": float(f1_score(y_test, y_pred, average="weighted")),
    "f1_macro": float(f1_score(y_test, y_pred, average="macro")),
}
print(json.dumps(metrics, indent=2))
print(classification_report(y_test, y_pred, target_names=iris.target_names))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

## 5. Save Artifacts and Metrics

Save the fitted estimator, test predictions, and JSON metrics. The DVC pipeline does this as separate stages; this cell demonstrates the same artifact outputs locally.

In [ ]:
artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
joblib.dump(model, artifact_dir / "iris_model.joblib")
pd.DataFrame({"actual": y_test, "predicted": y_pred}).to_csv(
    artifact_dir / "predictions.csv", index=False
)
(artifact_dir / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n", encoding="utf-8")
print("Saved model, predictions, and metrics under artifacts/.")

## 6. Convert Notebook Logic to Scripts

The notebook's data split, model fit, and evaluation logic is separated into `src/prepare.py`, `src/train.py`, and `src/evaluate.py`. Each has a command-line entry point so DVC and CI can invoke the same code without notebook state.

## 7. Parameterize with `params.yaml`

`params.yaml` controls `data.test_size`, random seeds, model type and hyperparameters, plus the evaluation accuracy threshold. DVC declares these parameter dependencies in `dvc.yaml`, so a parameter change invalidates only affected stages.

## 8. Initialize Git and DVC

`.gitignore` excludes Python/Jupyter caches and generated model/test outputs. Git tracks source and pipeline metadata; DVC tracks data, model outputs, and metrics. The repository is initialized in the project directory for this exercise.

## 9. Configure Remote and Track Data

The raw CSV is exported from scikit-learn's bundled dataset and tracked as `data/iris.csv.dvc`. A local DVC remote at `../dvc-storage` is configured for this classroom run; use a shared remote and external credentials for collaboration. `dvc push` verifies that the tracked dataset is stored there.

## 10. Create DVC Pipeline and Reproduce

`dvc.yaml` defines prepare → train → evaluate, including dependencies, parameter dependencies, generated artifacts, and the metrics file. Run `dvc repro` after creating the source CSV; DVC skips stages whose dependencies have not changed.

## 11. Compare Metrics and Parameters

After recording the baseline commit, change a parameter (for example `model.C`), rerun `dvc repro`, and compare working-tree results against `HEAD` with `dvc metrics diff` and `dvc params diff`. These commands report metric changes and parameter values without hand-copying results.

## 12. GitHub Actions + CML Workflow

`.github/workflows/cml.yaml` runs the pipeline for pull requests, regenerates Iris (no external download), and posts a CML comment containing current metrics plus DVC metrics/parameter diffs where a base revision is available. The workflow needs `pull-requests: write`; fork PR token restrictions may require maintainer approval or a different reporting strategy.